# Baseline generation on vLLM

Generates the three fixed-policy baselines (`never`, `always`, `adaptive`) over **all 1240**
single-turn BFCL items and writes JSONL to Google Drive. Scoring happens locally on CPU
afterwards — this notebook only produces raw generations.

**Runtime → Change runtime type → GPU** before starting. A T4 is sufficient.

### Why this runs here and not locally

HF `generate` batches statically: every sequence in a batch steps until the longest one
finishes. The local run spent 14229 decode steps to produce 96609 useful tokens — a 2.36x
waste factor. vLLM's continuous batching removes it. Local estimate for this work was ~16 h.

### Why the install is `--no-deps`

A full `bfcl-eval` install fights vLLM over the torch pin. Generation never needs the BFCL
*checker*, only the BFCL *data* — and `bfcl_eval/__init__.py` is empty, so `import bfcl_eval`
costs 0.002s while importing the checker costs 9.8s and 4322 modules. `bfcl_scorer` defers
that import, so `--no-deps` is enough here.

### Important

vLLM and HF `generate` do **not** produce byte-identical output even at temperature 0.
Do not mix runs from the two engines in one reported table — this notebook regenerates all
three policies wholesale so the resulting set is internally consistent.

## 1 — Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 2 — Install

Takes ~5–10 minutes. vLLM may replace Colab's preinstalled torch.

**If Colab shows a "RESTART SESSION" button when this finishes, click it, then resume from
step 3.** Everything from step 3 onward is idempotent and safe to re-run.

In [ ]:
!pip install -q vllm
# Data only. The full dependency tree pulls a conflicting torch, plus qwen_agent
# and soundfile, none of which generation touches.
!pip install -q --no-deps bfcl-eval==2026.3.23

## 3 — Verify the install survived, then get the code

Run this after any restart. If the `bfcl_eval` line fails, the `--no-deps` install did not
land; if the `vllm` line fails, restart the session first before reinstalling.

In [ ]:
import os

import bfcl_eval
import vllm

data_dir = os.path.join(os.path.dirname(bfcl_eval.__file__), "data")
print("vllm       :", vllm.__version__)
print("bfcl data  :", os.path.isdir(data_dir), data_dir)
assert os.path.isdir(data_dir), "BFCL data missing — rerun the --no-deps install"

In [ ]:
REPO = "https://github.com/widodu77/rs_aidams.git"

if os.path.isdir("/content/rs_aidams"):
    !cd /content/rs_aidams && git pull --ff-only
else:
    !git clone -q $REPO /content/rs_aidams

%cd /content/rs_aidams
os.environ["PYTHONPATH"] = "src"
!git log --oneline -1

## 4 — Mount Drive

Results are copied out after each policy, so a session drop costs at most one policy —
and `--resume` picks up mid-policy from whatever was already flushed.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Written to results/raw/vllm/, NOT results/raw/. The repo already tracks the
# HF-generated baselines under those exact filenames, and they are the provenance
# of every number in notes/2026-08-09.md. Since outputs from the two engines must
# not be mixed in one table, the two sets have to stay separately identifiable
# rather than one silently overwriting the other.
OUT_DIR = "results/raw/vllm"
DRIVE_OUT = "/content/drive/MyDrive/rs_aidams/results/raw/vllm"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# Restore anything from an earlier session so --resume can see it.
!cp -n $DRIVE_OUT/*.jsonl $OUT_DIR/ 2>/dev/null; ls -la $OUT_DIR/

## 5 — Generate

All 1240 items per policy — `simple_python` 400, `multiple` 200, `parallel` 200,
`parallel_multiple` 200, `irrelevance` 240. No `--limit`, so the 100-item cap that left the
`multiple` comparison underpowered (McNemar p=0.065) is gone.

`never` runs first: it is the cheapest and will surface any pipeline problem within minutes
rather than an hour into the expensive policy.

In [ ]:
for policy in ["never", "always", "adaptive"]:
    print(f"\n{'='*70}\n{policy}\n{'='*70}", flush=True)
    out = f"{OUT_DIR}/qwen3-1.7b_{policy}.jsonl"
    !python -m generate.run_vllm --policy $policy --out $out --resume
    !cp $out $DRIVE_OUT/
    print(f"{policy} copied to Drive", flush=True)

## 6 — Sanity check before leaving Colab

Cheap checks that catch a broken run *here*, rather than after downloading. The think-rate
check is the one that matters: if `never` is not ~0.0, the policy was not enforced and the
whole comparison is invalid.

In [ ]:
import json
import re
from collections import Counter

THINK = re.compile(r"<think>(.*?)</think>", re.S)

for policy in ["never", "always", "adaptive"]:
    path = f"{OUT_DIR}/qwen3-1.7b_{policy}.jsonl"
    if not os.path.exists(path):
        print(f"{policy:9s} MISSING")
        continue
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    per_cat = Counter(r["category"] for r in rows)
    # Matches the parser's rule: an empty <think></think> is not reasoning.
    thought = sum(
        1 for r in rows
        if (m := THINK.search(r["output_text"])) and m.group(1).strip()
    )
    trunc = sum(1 for r in rows if r["completion_tokens"] >= 768)
    print(
        f"{policy:9s} n={len(rows):5d}  think_rate={thought/len(rows):6.1%}  "
        f"trunc={trunc/len(rows):5.1%}  mean_tok={sum(r['completion_tokens'] for r in rows)/len(rows):6.1f}"
    )
    print(f"            {dict(per_cat)}")

print("\nexpected: never think_rate ~0.0, n=1240 per policy, trunc low")
print("if never is not ~0.0, the policy was not enforced — stop and diagnose")

## Then, locally

Copy the JSONL out of Drive into `results/raw/vllm/` in the repo, and score on CPU against
the full `bfcl-eval` install:

```bash
uv run python -m analysis.score_run results/raw/vllm/qwen3-1.7b_never.jsonl results/raw/vllm/qwen3-1.7b_always.jsonl results/raw/vllm/qwen3-1.7b_adaptive.jsonl --out results/baseline_metrics_vllm.json
```

The HF-generated files in `results/raw/` are left untouched, so the numbers currently in
`notes/2026-08-09.md` remain reproducible from the data that produced them.